In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics import median_absolute_error
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score, KFold
from scipy import stats

In [2]:
mlb_pitchers = pd.read_csv('Pitchers Cleaned.csv')
mlb_pitchers

,Player,Debut Year,Debut Age,Retirement Year,Retirement Age,Career Length,Wins,Losses,Win Percentage,Total Decisions,...,BF,ERA+,FIP,WHIP,H9,HR9,BB9,SO9,SO/BB,Hall of Fame
0,Cy Young,1890,23,1911,44,21,511,315,0.619,826,...,29565,138,2.84,1.130,8.7,0.2,1.5,3.4,2.30,1
1,Pud Galvin,1875,18,1892,35,17,365,310,0.541,675,...,25415,107,2.96,1.191,9.6,0.2,1.1,2.7,2.43,1
2,Walter Johnson,1907,19,1927,39,20,417,279,0.599,696,...,23415,147,2.38,1.061,7.5,0.1,2.1,5.3,2.57,1
3,Phil Niekro,1964,25,1987,48,23,318,274,0.537,592,...,22677,115,3.62,1.268,8.4,0.8,3.0,5.6,1.85,1
4,Nolan Ryan,1966,19,1993,46,27,324,292,0.526,616,...,22575,112,2.97,1.247,6.6,0.5,4.7,9.5,2.04,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1210,Charlie Robertson,1919,23,1928,32,9,49,80,0.380,129,...,4453,90,3.89,1.518,10.3,0.3,3.4,2.8,0.82,0
1211,Sammy Ellis,1962,21,1969,28,7,63,58,0.521,121,...,4296,88,3.90,1.340,8.7,1.1,3.4,6.1,1.79,0
1212,Lil Stoner,1922,23,1931,32,9,50,57,0.467,107,...,4466,87,4.13,1.548,10.6,0.6,3.4,2.7,0.80,0
1213,Johnny Humphries,1938,23,1946,31,8,52,63,0.452,115,...,4342,97,3.80,1.394,9.2,0.4,3.4,2.8,0.85,0


# Basic Version

In [26]:
# Define Model
dtree = DecisionTreeClassifier()

# Features
X = mlb_pitchers.drop(columns=['Hall of Fame', 'Player'])

# Target
y = mlb_pitchers['Hall of Fame']

In [27]:
# Split into training and testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

dtree = DecisionTreeClassifier()
dtree.fit(X_train, y_train)

DecisionTreeClassifier()

In [28]:
y_pred = dtree.predict(X_test)
y_pred

array([0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0,
       0])

In [29]:
dtree_init_results = pd.DataFrame()
dtree_init_results['Hall of Fame'] = y_test
dtree_init_results['Decision Tree Prediction'] = y_pred
dtree_init_results['Prediction Correct'] = (dtree_init_results['Hall of Fame'] == dtree_init_results['Decision Tree Prediction']).astype(int)
dtree_init_results

,Hall of Fame,Decision Tree Prediction,Prediction Correct
739,0,0,1
788,0,0,1
43,1,1,1
155,0,0,1
494,0,0,1
...,...,...,...
59,0,0,1
839,0,0,1
63,1,0,0
723,0,0,1


In [30]:
dtree_init_results['Prediction Correct'].value_counts()

Prediction Correct
1    234
0      9
Name: count, dtype: int64

# With K-Fold Regularization

In [14]:
dtree = DecisionTreeClassifier()

# Features
X = mlb_pitchers.drop(columns=['Hall of Fame', 'Player'])

# Target
y = mlb_pitchers['Hall of Fame']

# Split into training and testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

dtree = DecisionTreeClassifier()
dtree.fit(X_train, y_train)

DecisionTreeClassifier()

In [15]:
# Get predicted probabilities
y_proba_all = cross_val_predict(dtree, X, y, cv=kf, method='predict_proba')

# Regular predictions
y_pred_all = cross_val_predict(dtree, X, y, cv=kf)

# If binary classification, pick the probability of the predicted class
# (confidence in whichever class was chosen)
y_pred_indices = np.array([list(np.unique(y)).index(p) for p in y_pred_all])
confidence_scores = y_proba_all[np.arange(len(y_proba_all)), y_pred_indices]

In [16]:
from sklearn.model_selection import cross_val_predict

# Generate out-of-fold predictions for all data
kf = KFold(n_splits=5, shuffle=True, random_state=42)
y_pred_all = cross_val_predict(dtree, X, y, cv=kf)

# Build results dataframe
dtree_cv_results = pd.DataFrame()
dtree_cv_results['Player'] = mlb_pitchers['Player']
dtree_cv_results['Hall of Fame'] = y
dtree_cv_results['DTree Prediction'] = y_pred_all
dtree_cv_results['DTree Correct'] = (dtree_cv_results['Hall of Fame'] == dtree_cv_results['DTree Prediction']).astype(int)
dtree_cv_results['DTree Confidence'] = confidence_scores
dtree_cv_results

,Player,Hall of Fame,DTree Prediction,DTree Correct,DTree Confidence
0,Cy Young,1,0,0,1.0
1,Pud Galvin,1,0,0,1.0
2,Walter Johnson,1,1,1,1.0
3,Phil Niekro,1,0,0,0.0
4,Nolan Ryan,1,1,1,1.0
...,...,...,...,...,...
1210,Charlie Robertson,0,0,1,1.0
1211,Sammy Ellis,0,0,1,1.0
1212,Lil Stoner,0,0,1,1.0
1213,Johnny Humphries,0,0,1,1.0


In [17]:
dtree_cv_results['DTree Confidence'].value_counts()

DTree Confidence
1.0    1195
0.0      20
Name: count, dtype: int64

# Probability Fix

In [18]:
dtree = DecisionTreeClassifier()

# Features
X = mlb_pitchers.drop(columns=['Hall of Fame', 'Player'])

# Target
y = mlb_pitchers['Hall of Fame']

# Split into training and testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

dtree = DecisionTreeClassifier()
dtree.fit(X_train, y_train)

DecisionTreeClassifier()

In [19]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# get prob matrix (n_samples, n_classes)
y_proba_all = cross_val_predict(DecisionTreeClassifier(random_state=42),
                                X, y, cv=kf, method='predict_proba')

# classes are sorted like np.unique(y)
classes = np.unique(y)
print("Classes order:", classes)

# choose index of the positive class (change 1 to 'Yes' if needed)
pos_idx = np.where(classes == 1)[0][0]   # adapt if your positive label differs
pos_prob = y_proba_all[:, pos_idx]


Classes order: [0 1]


In [20]:
from sklearn.model_selection import cross_val_predict

# Generate out-of-fold predictions for all data
kf = KFold(n_splits=5, shuffle=True, random_state=42)
y_pred_all = cross_val_predict(dtree, X, y, cv=kf)

# Build results dataframe
dtree_cv_results = pd.DataFrame()
dtree_cv_results['Player'] = mlb_pitchers['Player']
dtree_cv_results['Hall of Fame'] = y
dtree_cv_results['DTree Prediction'] = y_pred_all
dtree_cv_results['DTree Correct'] = (dtree_cv_results['Hall of Fame'] == dtree_cv_results['DTree Prediction']).astype(int)
dtree_cv_results['DTree HOF Prob'] = pos_prob
dtree_cv_results

,Player,Hall of Fame,DTree Prediction,DTree Correct,DTree HOF Prob
0,Cy Young,1,1,1,0.0
1,Pud Galvin,1,0,0,0.0
2,Walter Johnson,1,1,1,1.0
3,Phil Niekro,1,1,1,1.0
4,Nolan Ryan,1,1,1,1.0
...,...,...,...,...,...
1210,Charlie Robertson,0,0,1,0.0
1211,Sammy Ellis,0,0,1,0.0
1212,Lil Stoner,0,0,1,0.0
1213,Johnny Humphries,0,0,1,0.0


In [ ]:
# Define K-Fold
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Cross-validation using R² score
cv_scores = cross_val_score(dtree, X, y, cv=kf, scoring='r2')

print("Cross-validation R² scores:", cv_scores)
print("Mean R²:", np.mean(cv_scores))
print("Standard deviation of R²:", np.std(cv_scores))